# Problem Statement 5: Data Analytics II – Logistic Regression
**Objective:** Implement Logistic Regression on the Social Network Ads dataset and compute the Confusion Matrix metrics.

**Dataset:** Social_Network_Ads.csv – Contains user information (Age, Estimated Salary) and whether they purchased a product (0/1).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, ConfusionMatrixDisplay
)

print("Libraries imported!")

## Step 1: Load the Dataset
We create a synthetic version of the Social Network Ads dataset since the original requires a Kaggle login.

In [ ]:
np.random.seed(42)
n = 400

age    = np.random.randint(18, 65, n)
salary = np.random.randint(15000, 150000, n)

# Simulate purchasing decision: older + higher salary -> more likely to buy
prob = 1 / (1 + np.exp(-(0.05 * age + 0.00002 * salary - 3.5)))
purchased = (np.random.rand(n) < prob).astype(int)

df = pd.DataFrame({
    'UserID': range(1, n + 1),
    'Gender': np.random.choice(['Male', 'Female'], n),
    'Age': age,
    'EstimatedSalary': salary,
    'Purchased': purchased
})

print("Dataset Shape:", df.shape)
print("Class distribution:")
print(df['Purchased'].value_counts())
df.head()

## Step 2: Prepare Features and Split Data

In [ ]:
X = df[['Age', 'EstimatedSalary']]
y = df['Purchased']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Feature scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

## Step 3: Train Logistic Regression Model

In [ ]:
classifier = LogisticRegression(random_state=42)
classifier.fit(X_train_sc, y_train)

y_pred = classifier.predict(X_test_sc)

print("Model trained and predictions made!")
print("\nSample predictions vs actual:")
print(pd.DataFrame({'Actual': y_test.values[:10], 'Predicted': y_pred[:10]}))

## Step 4: Confusion Matrix and Performance Metrics

In [ ]:
cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

print("Confusion Matrix:")
print(cm)
print(f"\nTP (True Positive)  = {TP}")
print(f"TN (True Negative)  = {TN}")
print(f"FP (False Positive) = {FP}")
print(f"FN (False Negative) = {FN}")

In [ ]:
accuracy   = (TP + TN) / (TP + TN + FP + FN)
error_rate = 1 - accuracy
precision  = TP / (TP + FP)
recall     = TP / (TP + FN)
f1         = 2 * precision * recall / (precision + recall)

print("Performance Metrics:")
print(f"  Accuracy   = {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"  Error Rate = {error_rate:.4f}  ({error_rate*100:.2f}%)")
print(f"  Precision  = {precision:.4f}")
print(f"  Recall     = {recall:.4f}")
print(f"  F1 Score   = {f1:.4f}")

In [ ]:
print("Detailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Purchased', 'Purchased']))

In [ ]:
# Visualize Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Purchased', 'Purchased'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix')

# Scatter plot of test data
X_test_arr = scaler.inverse_transform(X_test_sc)
scatter = axes[1].scatter(X_test_arr[:, 0], X_test_arr[:, 1], c=y_pred, cmap='coolwarm', alpha=0.7)
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Estimated Salary')
axes[1].set_title('Predictions on Test Set')
plt.colorbar(scatter, ax=axes[1], label='Predicted Class')

plt.tight_layout()
plt.show()